In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


# 1. Load dataset

df = pd.read_csv("home.csv")

# 2. Separate features and target

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

# 3. Remove ID variable

# SK_ID_CURR is an identifier, not a predictive feature

if "SK_ID_CURR" in X.columns:
    X = X.drop(columns=["SK_ID_CURR"])


# 4. Identify feature types

# Numerical features
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()


# All categorical features
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()


# Identify the ordinal variable
ordinal_features = ["NAME_EDUCATION_TYPE"]


# Remaining categorical variables are nominal
nominal_features = [
    col for col in categorical_features
    if col not in ordinal_features
]


print("Numerical features:", len(numerical_features))
print("Nominal features:", len(nominal_features))
print("Ordinal features:", len(ordinal_features))


# 5. Numerical preprocessing

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 6. Nominal categorical preprocessing

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

# 7. Ordinal categorical preprocessing

# Define the natural order of education levels.

education_order = [
    "Lower secondary",
    "Secondary / secondary special",
    "Incomplete higher",
    "Higher education"
]

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("encoder", OrdinalEncoder(
        categories=[education_order],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# 8. Combine all preprocessing

preprocessor = ColumnTransformer([
    
    ("numerical",
     numerical_pipeline,
     numerical_features),

    ("nominal",
     nominal_pipeline,
     nominal_features),

    ("ordinal",
     ordinal_pipeline,
     ordinal_features)
])

# 9. Create complete model pipeline

model = Pipeline([
    
    ("preprocessing", preprocessor),

    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])


# 10. Split data into training and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# 11. Train complete pipeline

model.fit(X_train, y_train)

# 12. Evaluate on held-out test data

y_pred = model.predict(X_test)

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

C:\Users\user\AppData\Local\Temp\ipykernel_21636\2500927691.py:48: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


Numerical features: 104
Nominal features: 15
Ordinal features: 1

Accuracy:
0.919337268100743

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.83      0.00      0.00      4965

    accuracy                           0.92     61503
   macro avg       0.88      0.50      0.48     61503
weighted avg       0.91      0.92      0.88     61503



In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# 1. Load data
df = pd.read_csv("prices.csv")

# Remove ID column
X = df.drop(columns=["SalePrice", "Id"])
y = df["SalePrice"]


# 2. Identify numerical and categorical features
num_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

cat_features = X.select_dtypes(
    include=["object"]
).columns


# 3. Numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# 4. Categorical preprocessing
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# 5. Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])


# 6. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# 7. OLS
ols_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])


# 8. Ridge with cross-validation
ridge_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Ridge())
])

ridge_grid = GridSearchCV(
    ridge_pipeline,
    {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=5,
    scoring="neg_mean_squared_error"
)


# 9. Lasso with cross-validation
lasso_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", Lasso(max_iter=10000))
])

lasso_grid = GridSearchCV(
    lasso_pipeline,
    {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=5,
    scoring="neg_mean_squared_error"
)


# 10. Fit models
ols_pipeline.fit(X_train, y_train)
ridge_grid.fit(X_train, y_train)
lasso_grid.fit(X_train, y_train)


# 11. Generate predictions
models = {
    "OLS": ols_pipeline,
    "Ridge": ridge_grid.best_estimator_,
    "Lasso": lasso_grid.best_estimator_
}


# 12. Evaluate models
for name, model in models.items():

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n{name}")
    print(f"MAE  : {mae:.6f}")
    print(f"RMSE : {rmse:.6f}")
    print(f"R²   : {r2:.6f}")


print("\nBest Ridge alpha:", ridge_grid.best_params_)
print("Best Lasso alpha:", lasso_grid.best_params_)

C:\Users\user\AppData\Local\Temp\ipykernel_28076\2109268694.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(
C:\Users\user\anaconda3\envs\NijasAI\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:784: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.389017e+10, tolerance: 5.374e+08
  model = cd_fast.sparse_enet_coordinate_descent(
C:\Users\user\anaconda3\envs\NijasAI\Lib\site-packages\sklea


OLS
MAE  : 18287.692072
RMSE : 29474.114694
R²   : 0.886742

Ridge
MAE  : 19040.551135
RMSE : 30646.233538
R²   : 0.877555

Lasso
MAE  : 17095.196123
RMSE : 28374.355196
R²   : 0.895037

Best Ridge alpha: {'model__alpha': 10}
Best Lasso alpha: {'model__alpha': 100}
